# Preprocesado de Imágenes

1. **DenseNet121** 
2. **SVD** sobre features de imagen
3. **Features de calidad** (blur, brightness)
4. **Procesar múltiples imágenes** (1ra y 2da por separado)
5. **CLIP** (moderno, multimodal)

In [1]:
!pip install -q transformers

In [2]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from collections import defaultdict
import cv2

import torch
from transformers import CLIPProcessor, CLIPModel
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.utils import load_img, img_to_array
from sklearn.decomposition import TruncatedSVD

import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")

2025-11-29 22:30:55.136281: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764455455.354095      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764455455.416105      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

✅ Librerías importadas


In [3]:
# Configuración
TRAIN_IMG_PATH = '../input/petfinder-adoption-prediction/train_images'
TEST_IMG_PATH = '../input/petfinder-adoption-prediction/test_images'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

train = pd.read_csv('../input/petfinder-adoption-prediction/train/train.csv').set_index("PetID")
test = pd.read_csv('../input/petfinder-adoption-prediction/test/test.csv').set_index("PetID")

print(f"Train: {len(train)}, Test: {len(test)}")

Train: 14993, Test: 3972


In [4]:
def get_pet_images(img_path, pet_ids):
    pet_images = defaultdict(list)
    all_images = set(os.listdir(img_path))
    for pet_id in pet_ids:
        pet_imgs = sorted([f for f in all_images if f.startswith(pet_id)])
        pet_images[pet_id] = pet_imgs
    return pet_images

train_pet_images = get_pet_images(TRAIN_IMG_PATH, train.index.tolist())
test_pet_images = get_pet_images(TEST_IMG_PATH, test.index.tolist())

## 1. Features de Metadatos + Calidad de Imagen

In [5]:
def calculate_blur(img_array):
    """Calcula el blur score usando Laplacian variance."""
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def extract_image_metadata(img_path, pet_images):
    features = []
    for pet_id, images in tqdm(pet_images.items(), desc="Metadatos"):
        feat = {
            'img_count': len(images),
            'img_has_photo': 1 if len(images) > 0 else 0,
        }
        
        if len(images) > 0:
            try:
                img = Image.open(os.path.join(img_path, images[0]))
                img_array = np.array(img.convert('RGB'))
                
                feat['img_width'] = img.size[0]
                feat['img_height'] = img.size[1]
                feat['img_aspect_ratio'] = img.size[0] / max(1, img.size[1])
                feat['img_size'] = img.size[0] * img.size[1]
                feat['img_mean_brightness'] = img_array.mean()
                feat['img_std_brightness'] = img_array.std()
                
                # Blur score (técnica de ganadores)
                feat['img_blur_score'] = calculate_blur(img_array)
                
                # Color dominance
                feat['img_red_mean'] = img_array[:,:,0].mean()
                feat['img_green_mean'] = img_array[:,:,1].mean()
                feat['img_blue_mean'] = img_array[:,:,2].mean()
            except:
                for k in ['img_width', 'img_height', 'img_aspect_ratio', 'img_size', 
                          'img_mean_brightness', 'img_std_brightness', 'img_blur_score',
                          'img_red_mean', 'img_green_mean', 'img_blue_mean']:
                    feat[k] = 0
        else:
            for k in ['img_width', 'img_height', 'img_aspect_ratio', 'img_size', 
                      'img_mean_brightness', 'img_std_brightness', 'img_blur_score',
                      'img_red_mean', 'img_green_mean', 'img_blue_mean']:
                feat[k] = 0
        
        features.append(feat)
    
    df = pd.DataFrame(features)
    df.index = list(pet_images.keys())
    return df

train_meta = extract_image_metadata(TRAIN_IMG_PATH, train_pet_images)
test_meta = extract_image_metadata(TEST_IMG_PATH, test_pet_images)
print(f"Metadatos: {train_meta.shape[1]} features")

Metadatos: 100%|██████████| 3972/3972 [00:46<00:00, 85.23it/s] 

Metadatos: 12 features


## 2. DenseNet121 Features

In [6]:
# Cargar DenseNet121 
densenet = DenseNet121(weights='imagenet', include_top=False, pooling='avg')
print(f"DenseNet121 cargado. Output: {densenet.output_shape}")

I0000 00:00:1764455799.849184      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1764455799.849836      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
DenseNet121 cargado. Output: (None, 1024)


In [7]:
def extract_densenet_features(img_path, pet_images, model, img_size=(224, 224)):
    all_features_1st = []  # Primera imagen
    all_features_2nd = []  # Segunda imagen (si existe)
    pet_ids = []
    output_dim = model.output_shape[-1]  # 1024
    
    for pet_id, images in tqdm(pet_images.items(), desc="DenseNet"):
        # Primera imagen
        if len(images) >= 1:
            try:
                img = load_img(os.path.join(img_path, images[0]), target_size=img_size)
                img_array = preprocess_input(np.expand_dims(img_to_array(img), axis=0))
                feat_1 = model.predict(img_array, verbose=0).flatten()
            except:
                feat_1 = np.zeros(output_dim)
        else:
            feat_1 = np.zeros(output_dim)
        
        # Segunda imagen 
        if len(images) >= 2:
            try:
                img = load_img(os.path.join(img_path, images[1]), target_size=img_size)
                img_array = preprocess_input(np.expand_dims(img_to_array(img), axis=0))
                feat_2 = model.predict(img_array, verbose=0).flatten()
            except:
                feat_2 = np.zeros(output_dim)
        else:
            feat_2 = np.zeros(output_dim)
        
        all_features_1st.append(feat_1)
        all_features_2nd.append(feat_2)
        pet_ids.append(pet_id)
    
    return np.array(all_features_1st), np.array(all_features_2nd), pet_ids

print("Extrayendo DenseNet features...")
train_dn_1st, train_dn_2nd, train_ids = extract_densenet_features(TRAIN_IMG_PATH, train_pet_images, densenet)
test_dn_1st, test_dn_2nd, test_ids = extract_densenet_features(TEST_IMG_PATH, test_pet_images, densenet)

print(f"DenseNet 1st shape: {train_dn_1st.shape}")
print(f"DenseNet 2nd shape: {train_dn_2nd.shape}")

Extrayendo DenseNet features...


DenseNet:   0%|          | 0/14993 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1764455806.825070     105 service.cc:148] XLA service 0x7fb5e40b43e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1764455806.826192     105 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1764455806.826230     105 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1764455808.095910     105 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1764455814.562080     105 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
DenseNet: 100%|██████████| 3972/3972 [09:47<00:00,  6.76it/s]

DenseNet 1st shape: (14993, 1024)
DenseNet 2nd shape: (14993, 1024)


In [8]:
# SVD sobre DenseNet features
N_SVD = 32

# SVD para primera imagen
svd_1st = TruncatedSVD(n_components=N_SVD, random_state=42)
train_dn1_svd = svd_1st.fit_transform(train_dn_1st)
test_dn1_svd = svd_1st.transform(test_dn_1st)

# SVD para segunda imagen
svd_2nd = TruncatedSVD(n_components=N_SVD, random_state=42)
train_dn2_svd = svd_2nd.fit_transform(train_dn_2nd)
test_dn2_svd = svd_2nd.transform(test_dn_2nd)

train_dn1_df = pd.DataFrame(train_dn1_svd, columns=[f'dn1_svd_{i}' for i in range(N_SVD)], index=train_ids)
test_dn1_df = pd.DataFrame(test_dn1_svd, columns=[f'dn1_svd_{i}' for i in range(N_SVD)], index=test_ids)

train_dn2_df = pd.DataFrame(train_dn2_svd, columns=[f'dn2_svd_{i}' for i in range(N_SVD)], index=train_ids)
test_dn2_df = pd.DataFrame(test_dn2_svd, columns=[f'dn2_svd_{i}' for i in range(N_SVD)], index=test_ids)

print(f"DenseNet SVD: {N_SVD * 2} features totales")

DenseNet SVD: 64 features totales


## 3. CLIP Features (Moderno - Multimodal)

In [9]:
# CLIP para features semánticas
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("CLIP cargado")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP cargado


In [10]:
def extract_clip_features(img_path, pet_images, model, processor, device):
    all_features = []
    pet_ids = []
    
    for pet_id, images in tqdm(pet_images.items(), desc="CLIP"):
        if len(images) == 0:
            all_features.append(np.zeros(512))
            pet_ids.append(pet_id)
            continue
        
        # Solo primera imagen para CLIP
        try:
            img = Image.open(os.path.join(img_path, images[0])).convert('RGB')
            inputs = processor(images=img, return_tensors="pt").to(device)
            with torch.no_grad():
                features = model.get_image_features(**inputs)
                features = features / features.norm(dim=-1, keepdim=True)
            all_features.append(features.cpu().numpy().flatten())
        except:
            all_features.append(np.zeros(512))
        
        pet_ids.append(pet_id)
    
    return np.array(all_features), pet_ids

train_clip, train_clip_ids = extract_clip_features(TRAIN_IMG_PATH, train_pet_images, clip_model, clip_processor, DEVICE)
test_clip, test_clip_ids = extract_clip_features(TEST_IMG_PATH, test_pet_images, clip_model, clip_processor, DEVICE)

print(f"CLIP shape: {train_clip.shape}")

CLIP: 100%|██████████| 3972/3972 [00:55<00:00, 71.24it/s]

CLIP shape: (14993, 512)


In [11]:
# SVD sobre CLIP
N_CLIP_SVD = 32

svd_clip = TruncatedSVD(n_components=N_CLIP_SVD, random_state=42)
train_clip_svd = svd_clip.fit_transform(train_clip)
test_clip_svd = svd_clip.transform(test_clip)

train_clip_df = pd.DataFrame(train_clip_svd, columns=[f'clip_svd_{i}' for i in range(N_CLIP_SVD)], index=train_clip_ids)
test_clip_df = pd.DataFrame(test_clip_svd, columns=[f'clip_svd_{i}' for i in range(N_CLIP_SVD)], index=test_clip_ids)

print(f"CLIP SVD: {N_CLIP_SVD} features")

CLIP SVD: 32 features


## 4. Combinar y Guardar

In [12]:
# Combinar todo
train_features = train_meta.join(train_dn1_df).join(train_dn2_df).join(train_clip_df)
test_features = test_meta.join(test_dn1_df).join(test_dn2_df).join(test_clip_df)

# Reindexar
train_features = train_features.reindex(train.index).fillna(0)
test_features = test_features.reindex(test.index).fillna(0)

print(f"\n✅ Total features de imagen: {train_features.shape[1]}")


✅ Total features de imagen: 108


In [13]:
# Guardar
train_features.to_parquet("train.parquet")
test_features.to_parquet("test.parquet")

print(f"Guardado: train.parquet {train_features.shape}")
print(f"Guardado: test.parquet {test_features.shape}")

Guardado: train.parquet (14993, 108)
Guardado: test.parquet (3972, 108)
